# train_walker2d_sac.ipynb
**SAC Robustness Check — Walker2d-v4**   
Conditions: M1, M2, M3 | Seeds: 5 | Budget: 1,000,000 steps/seed/condition  

SAC uses n_envs=1 (off-policy, replay buffer) — no VecEnv wrapper required.  
Network: [256, 256] ReLU + Adam, consistent with PPO branch.  

**Walker2d-v4 failure criterion** (env default, do NOT override):  
  `terminated=True` when z ∉ [0.8, 2.0] OR |angle| > 1.0  
  (distinct from Hopper's z < 0.7 threshold — never substitute)

In [1]:
# ── Mount Google Drive for persistent output ───────────────────────────────────
# Colab runtime storage is wiped on disconnect — save to Drive.
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/thesis_results/mujoco/walker2d"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Mounted at /content/drive
Output directory: /content/drive/MyDrive/thesis_results/mujoco/walker2d


In [2]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install stable-baselines3[extra] gymnasium[mujoco] --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 58.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 17.6 MB/s eta 0:00:00


In [3]:
import time
import numpy as np
import torch
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import SAC

torch.set_num_threads(os.cpu_count())

# ── Config ─────────────────────────────────────────────────────────────────────
ENV_ID      = "Walker2d-v4"
TOTAL_STEPS = 1_000_000
SEEDS       = [0, 1, 2, 3, 4]

# Walker2d-v4 failure criterion (env default, never override):
#   healthy_z_range=(0.8, 2.0), healthy_angle_range=(-1.0, 1.0)
#   DO NOT use Hopper's z < 0.7 threshold here.
CONDITIONS = {
    "M1": {"timelimit": 1000, "reset_noise_scale": 0.005},
    "M2": {"timelimit": 250,  "reset_noise_scale": 0.005},
    "M3": {"timelimit": 1000, "reset_noise_scale": 0.02},
}

# Network locked to [256,256] ReLU Adam — consistent with PPO branch (v12.2 §3.4.5)
SAC_KWARGS = dict(
    learning_rate    = 3e-4,
    buffer_size      = 1_000_000,
    learning_starts  = 10_000,
    batch_size       = 256,
    tau              = 0.005,
    gamma            = 0.99,
    train_freq       = 1,
    gradient_steps   = 1,
    ent_coef         = "auto",
    target_entropy   = "auto",
    use_sde          = False,
    policy_kwargs    = dict(
        net_arch        = [256, 256],
        activation_fn   = torch.nn.ReLU,
        optimizer_class = torch.optim.Adam,
    ),
    verbose          = 1,
    device           = "cpu",
)

print(f"Output: {OUTPUT_DIR}")
print(f"Conditions: {list(CONDITIONS.keys())} | Seeds: {SEEDS}")

Output: /content/drive/MyDrive/thesis_results/mujoco/walker2d
Conditions: ['M1', 'M2', 'M3'] | Seeds: [0, 1, 2, 3, 4]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# ── Environment factory ────────────────────────────────────────────────────────
def make_env(timelimit, reset_noise_scale, seed):
    """Single env, no VecEnv — SAC is off-policy."""
    env = gym.make(
        ENV_ID,
        reset_noise_scale=reset_noise_scale,
        # healthy_z_range and healthy_angle_range intentionally not passed;
        # Walker2d-v4 env defaults (0.8, 2.0) and (-1.0, 1.0) apply.
    )
    env = TimeLimit(env, max_episode_steps=timelimit)
    env.reset(seed=seed)
    return env

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────
# Skip-if-exists: safe to re-run after disconnect.
print("=" * 60)
print(f"SAC TRAINING — {ENV_ID}")
print(f"Steps/run: {TOTAL_STEPS:,} | n_envs: 1 (SAC off-policy) | Device: cpu")
print("=" * 60)

total_start = time.time()

for cond_name, cond_cfg in CONDITIONS.items():
    print(f"\n── Condition {cond_name} | TimeLimit={cond_cfg['timelimit']} | "
          f"noise={cond_cfg['reset_noise_scale']} ──")

    for seed in SEEDS:
        save_path = os.path.join(OUTPUT_DIR, f"{cond_name}_sac_seed{seed}.zip")

        if os.path.exists(save_path):
            print(f"  [skip] {cond_name} seed={seed} — already trained")
            continue

        run_start = time.time()
        print(f"  [train] {cond_name} seed={seed} ...", flush=True)

        env   = make_env(cond_cfg["timelimit"], cond_cfg["reset_noise_scale"], seed)
        model = SAC("MlpPolicy", env, seed=seed, **SAC_KWARGS)
        model.learn(total_timesteps=TOTAL_STEPS)
        model.save(save_path)
        env.close()

        elapsed = time.time() - run_start
        print(f"  [done]  {cond_name} seed={seed} → {save_path} ({elapsed/60:.1f} min)")

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"All training complete in {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} h)")
print(f"{'='*60}")

SAC TRAINING — Walker2d-v4
Steps/run: 1,000,000 | n_envs: 1 (SAC off-policy) | Device: cpu

── Condition M1 | TimeLimit=1000 | noise=0.005 ──
  [train] M1 seed=0 ...


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Walker2d-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 34       |
|    ep_rew_mean     | 10.4     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1728     |
|    time_elapsed    | 0        |
|    total_timesteps | 136      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 28       |
|    ep_rew_mean     | 5.48     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 1699     |
|    time_elapsed    | 0        |
|    total_timesteps | 224      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 25.3     |
|    ep_rew_mean     | 4.88     |
| time/              |          |
|    episodes        | 12       |
|    fps             |

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21       |
|    ep_rew_mean     | 2.41     |
| time/              |          |
|    episodes        | 24       |
|    fps             | 1655     |
|    time_elapsed    | 0        |
|    total_timesteps | 505      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.5     |
|    ep_rew_mean     | 2.45     |
| time/              |          |
|    episodes        | 28       |
|    fps             | 1651     |
|    time_elapsed    | 0        |
|    total_timesteps | 603      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.3     |
|    ep_rew_mean     | 2.21     |
| time/              |          |
|    episodes        | 32       |
|    fps             | 1624     |
|    time_elapsed    | 0        |
|    total_timesteps | 683      |
--------------

In [ ]:
# ── Inventory ──────────────────────────────────────────────────────────────────
print("Output inventory:")
for cond_name in CONDITIONS:
    for seed in SEEDS:
        path   = os.path.join(OUTPUT_DIR, f"{cond_name}_sac_seed{seed}.zip")
        status = "✓" if os.path.exists(path) else "✗ MISSING"
        print(f"  {status}  {cond_name}_sac_seed{seed}.zip")